# Maximal Spec Tau8 Handoff

Standalone restart-safe notebook for the post-500U checkpoint.

What running all cells does:

1. Mount Drive and refresh `/content/dismantle` to `codex/maximal-spec-colab`.
2. Mirror the already-finished 500U leaderboard/winner artifacts into a separate `dismantle_export` folder.
3. Warm-start tau8 rollout fine-tunes from the leaderboard winner into new `tau8_*` checkpoint folders.
4. Evaluate tau/frontier and merge those rows back into `leaderboard.json`.
5. Run a best-head continuation ladder that repeatedly warm-starts from the current leaderboard winner.
6. Mirror the updated artifacts again.

The export cells copy files only. They do not mutate trained checkpoint directories.


In [ ]:
# Cell 1 - Restart-safe setup: Drive, repo, packages, GPU

from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile

REPO_URL = 'https://github.com/joshuahickscorp/dismantle.git'
BRANCH = 'codex/maximal-spec-colab'
REPO_DIR = Path('/content/dismantle')
DRIVE_ROOT = Path('/content/drive/MyDrive/dismantle')
LAB_ROOT = DRIVE_ROOT / 'maximal_spec_500u'
EXPORT_ROOT = DRIVE_ROOT / 'dismantle_export' / 'maximal_spec_500u'

# Set this False if you only want the safe export and do not want extra training.
RUN_TAU8 = True
RUN_TAU8_EVAL = True
RUN_TRAIN_ON_BEST = True
RUN_TRAIN_ON_BEST_EVAL = True
BEST_CONTINUE_ROUNDS = 2
SAFE_EXPORT_ZIP = False
SAFE_EXPORT_TOP_N = 30
SAFE_EXPORT_HEAD_TOP_N = 3

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception as e:
    print(f'[setup] Drive mount skipped/failed: {e}')


def run(cmd, *, cwd=None):
    print('$', ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)

if not (REPO_DIR / '.git').exists():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR])
else:
    run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH, '--depth', '1'])
    run(['git', '-C', REPO_DIR, 'checkout', BRANCH])
    run(['git', '-C', REPO_DIR, 'reset', '--hard', f'origin/{BRANCH}'])

os.chdir(REPO_DIR)
HEAD_SHA = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print(f'[setup] repo={REPO_DIR} branch={BRANCH} sha={HEAD_SHA}')

run([sys.executable, '-u', '-m', 'pip', 'install', '-q', 'pyarrow>=17', 'tqdm>=4.66', 'zstandard', 'safetensors>=0.4'])

import numpy as np
import torch

assert torch.cuda.is_available(), 'No CUDA device. Runtime > Change runtime type > GPU.'
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
BIG_GPU = VRAM_GB >= 40
print(json.dumps({'gpu': GPU_NAME, 'vram_gb': VRAM_GB, 'big_gpu': BIG_GPU, 'repo_sha': HEAD_SHA}, indent=2))


In [ ]:
# Cell 2 - Helpers, target config, safe export function

from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
import time
import hashlib
import zipfile

QWEN_LOCKED_ENV = {
    'DISMANTLE_QWEN_TCB': '1',
    'DISMANTLE_QWEN_VOCAB_PRUNE': '32000',
    'DISMANTLE_QWEN_Q4K_LMHEAD': '1',
    'DISMANTLE_QWEN_FFN_DOWN_Q4K': '1',
    'DISMANTLE_QWEN_Q4K_PREDEC': '1',
}

TARGETS = {
    'q1p5': {
        'model_id': 'Qwen/Qwen2.5-1.5B-Instruct',
        'artifact_dir': LAB_ROOT / 'artifacts' / 'q1p5',
        'corpus_dir': LAB_ROOT / 'corpora' / 'q1p5_corpus',
        'drive_frozen': LAB_ROOT / 'artifacts' / 'q1p5' / 'q1p5_frozen.npz',
        'local_frozen': Path('/content/q1p5_frozen.npz'),
        'capture_layer': 24,
        'train_max_rows': 24000 if BIG_GPU else 8000,
        'train_max_row_tokens': 384 if BIG_GPU else 192,
        'eval_max_windows': 24000 if BIG_GPU else 6000,
        'frontier_max_depth': 24 if BIG_GPU else 12,
        'base_tps_placeholder': 95.0,
        'spec_efficiency_placeholder': 0.80,
        'gguf_name': 'qwen2.5-1.5b-instruct-q4_k_m.gguf',
        'profile_name': 'qwen15b-instruct-q4k.m3pro18.json',
    },
}

TAU_DEPTH = 8
FRONTIER_DEPTHS = '2,4,6,8,12,16,24'
FRONTIER_WIDTHS = '2,3,4,6,8'
TAU8_TARGET_ORDER = ['q1p5']
TAU8_TOP_BASELINES_PER_TARGET = 1

TAU8_SPECS = [
    {
        'name': 'rollout_d5_w015_p075_g090_lr1e-4',
        'epochs': 4,
        'lr': 1e-4,
        'rollout_loss_weight': 0.15,
        'rollout_depth': 5,
        'rollout_starts_per_batch': 4,
        'rollout_draft_prob': 0.75,
        'rollout_depth_gamma': 0.90,
        'calib_loss_weight': 0.12,
        'residual_delta_loss_weight': 0.01,
    },
    {
        'name': 'rollout_d8_w010_p050_g090_lr8e-5',
        'epochs': 4,
        'lr': 8e-5,
        'rollout_loss_weight': 0.10,
        'rollout_depth': 8,
        'rollout_starts_per_batch': 2,
        'rollout_draft_prob': 0.50,
        'rollout_depth_gamma': 0.90,
        'calib_loss_weight': 0.12,
        'residual_delta_loss_weight': 0.015,
    },
    {
        'name': 'rollout_d5_hardneg_w020_p100_lr8e-5',
        'epochs': 3,
        'lr': 8e-5,
        'rollout_loss_weight': 0.20,
        'rollout_depth': 5,
        'rollout_starts_per_batch': 4,
        'rollout_draft_prob': 1.00,
        'rollout_depth_gamma': 0.85,
        'calib_loss_weight': 0.16,
        'residual_delta_loss_weight': 0.02,
    },
]


def load_json(path, default):
    path = Path(path)
    try:
        return json.loads(path.read_text())
    except FileNotFoundError:
        return default


def write_json_atomic(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + '.tmp')
    tmp.write_text(json.dumps(payload, indent=2, sort_keys=True) + '\n')
    os.replace(tmp, path)


def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def remount_drive_for_export():
    try:
        from google.colab import drive
        print('[export] remounting Drive after copy failure...')
        try:
            drive.flush_and_unmount()
            time.sleep(2)
        except Exception as e:
            print(f'[export] flush/unmount skipped: {e}')
        drive.mount('/content/drive', force_remount=True)
        time.sleep(2)
    except Exception as e:
        print(f'[export] Drive remount failed: {e}')


def copy_file(src, dst, key, manifest, sha=False, retries=3):
    src = Path(src)
    dst = Path(dst)
    manifest.setdefault('copy_errors', {})
    last_error = None
    for attempt in range(1, retries + 1):
        tmp = dst.with_suffix(dst.suffix + f'.tmp.{attempt}')
        try:
            if not src.exists() or not src.is_file():
                manifest['missing'][key] = str(src)
                return None
            dst.parent.mkdir(parents=True, exist_ok=True)
            if tmp.exists():
                tmp.unlink()
            # Avoid shutil.copy2's sendfile fast path; Drive sometimes drops
            # that transport for large safetensors. Plain streaming is slower
            # but much more reliable, and retry/remount keeps export going.
            with open(src, 'rb') as fsrc, open(tmp, 'wb') as fdst:
                shutil.copyfileobj(fsrc, fdst, length=16 * 1024 * 1024)
                fdst.flush()
                os.fsync(fdst.fileno())
            shutil.copystat(src, tmp)
            os.replace(tmp, dst)
            info = {'source': str(src), 'exported': str(dst), 'bytes': int(dst.stat().st_size)}
            if sha:
                info['sha256'] = sha256_file(dst)
            manifest['files'][key] = info
            manifest['copy_errors'].pop(key, None)
            return info
        except OSError as e:
            last_error = repr(e)
            print(f'[export] copy failed attempt {attempt}/{retries}: {src} -> {dst}: {e}')
            try:
                if tmp.exists():
                    tmp.unlink()
            except Exception:
                pass
            if getattr(e, 'errno', None) == 107 or 'Transport endpoint is not connected' in str(e):
                remount_drive_for_export()
            time.sleep(min(20, 2 * attempt))
    manifest['missing'][key] = str(src)
    manifest['copy_errors'][key] = last_error or 'unknown copy error'
    print(f'[export] giving up on optional copy key={key}; export will continue')
    return None


def zip_dir(src_dir, zip_path):
    src_dir = Path(src_dir)
    zip_path = Path(zip_path)
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for path in sorted(src_dir.rglob('*')):
            if path.is_file() and path != zip_path:
                zf.write(path, path.relative_to(src_dir))


def run_with_heartbeat(cmd, label='job', interval_sec=60):
    print('$', ' '.join(map(str, cmd)), flush=True)
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    start = time.time()
    last = start
    assert p.stdout is not None
    for line in p.stdout:
        print(line, end='')
        now = time.time()
        if now - last >= interval_sec:
            mem = 'n/a'
            gpu = 'n/a'
            try:
                out = subprocess.check_output([
                    'nvidia-smi', '--query-gpu=utilization.gpu,memory.used,memory.total',
                    '--format=csv,noheader,nounits'
                ], text=True).strip().splitlines()[0].split(',')
                gpu = out[0].strip() + '%'
                mem = out[1].strip() + '/' + out[2].strip() + 'MB'
            except Exception:
                pass
            print(f'[{label}] RUNNING elapsed={(now-start)/60:.1f}m gpu={gpu} mem={mem}', flush=True)
            last = now
    rc = p.wait()
    print(f'[{label}] finished rc={rc} elapsed={(time.time()-start)/60:.1f}m', flush=True)
    if rc != 0:
        raise RuntimeError(f'{label} failed with rc={rc}')


def safe_export(note='pre_tau8'):
    leaderboard_path = LAB_ROOT / 'leaderboard.json'
    leaderboard = load_json(leaderboard_path, {'rows': []})
    rows = leaderboard.get('rows', []) if isinstance(leaderboard, dict) else []
    rows = sorted(
        rows,
        key=lambda r: (float(r.get('offline_projected_tps') or 0.0), float(r.get('tau') or 0.0)),
        reverse=True,
    )
    manifest = {
        'schema': 'dismantle-maximal-spec-tau8-export-v1',
        'created_at_unix': int(time.time()),
        'repo_sha': HEAD_SHA,
        'note': note,
        'lab_root': str(LAB_ROOT),
        'export_root': str(EXPORT_ROOT),
        'top_n': SAFE_EXPORT_TOP_N,
        'files': {},
        'missing': {},
        'leaderboard_rows': [],
    }
    meta_dir = EXPORT_ROOT / 'metadata'
    copy_file(leaderboard_path, meta_dir / 'leaderboard.json', 'leaderboard', manifest)
    for name in ['maximal_spec_summary.md', 'maximal_spec_summary.json']:
        copy_file(LAB_ROOT / name, meta_dir / name, name, manifest)
    for progress_path in sorted(LAB_ROOT.glob('progress*.json')):
        copy_file(progress_path, meta_dir / progress_path.name, f'progress/{progress_path.name}', manifest)

    for rank, row in enumerate(rows[:SAFE_EXPORT_TOP_N], start=1):
        target = str(row.get('target') or 'unknown')
        tag = str(row.get('tag') or f'rank_{rank}')
        safe_tag = ''.join(c if c.isalnum() or c in '._-+' else '_' for c in tag)
        manifest['leaderboard_rows'].append({
            'rank': rank,
            'target': target,
            'tag': tag,
            'tau': row.get('tau'),
            'accepted_draft_tokens_per_verify': row.get('accepted_draft_tokens_per_verify'),
            'offline_projected_tps': row.get('offline_projected_tps'),
            'policy_kind': row.get('policy_kind'),
        })
        if row.get('head') and rank <= SAFE_EXPORT_HEAD_TOP_N:
            copy_file(row['head'], EXPORT_ROOT / 'heads' / target / f'{rank:02d}_{safe_tag}.safetensors', f'heads/{target}/{tag}', manifest, sha=True)
        for key in ['tau_path', 'frontier_path']:
            if row.get(key):
                copy_file(row[key], EXPORT_ROOT / 'eval' / target / safe_tag / Path(row[key]).name, f'eval/{target}/{tag}/{Path(row[key]).name}', manifest)

    for target_dir in sorted(p for p in (LAB_ROOT / 'artifacts').glob('*') if p.is_dir()):
        for artifact in sorted(target_dir.glob('*')):
            if artifact.is_file() and artifact.suffix.lower() in {'.json', '.npz'}:
                copy_file(artifact, EXPORT_ROOT / 'artifacts' / target_dir.name / artifact.name, f'artifacts/{target_dir.name}/{artifact.name}', manifest)

    manifest_path = EXPORT_ROOT / 'manifest.json'
    write_json_atomic(manifest_path, manifest)
    if SAFE_EXPORT_ZIP:
        zip_path = EXPORT_ROOT.with_suffix('.zip')
        zip_dir(EXPORT_ROOT, zip_path)
        manifest['zip_path'] = str(zip_path)
        manifest['zip_bytes'] = int(zip_path.stat().st_size)
        write_json_atomic(manifest_path, manifest)

    total = sum(v['bytes'] for v in manifest['files'].values())
    print(f'[export:{note}] root={EXPORT_ROOT}')
    print(f'[export:{note}] copied={len(manifest["files"])} missing={len(manifest["missing"])} total={total/1e9:.2f}GB manifest={manifest_path}')
    for r in manifest['leaderboard_rows'][:10]:
        print(f"  #{r['rank']:02d} {r['target']} {r['tag']} tau={r.get('tau')} accepted={r.get('accepted_draft_tokens_per_verify')} tps={r.get('offline_projected_tps')}")
    return manifest


In [ ]:
# Cell 3 - Safe export of the completed 500U run before any new training

assert (LAB_ROOT / 'leaderboard.json').exists(), f'Missing leaderboard: {LAB_ROOT / "leaderboard.json"}'
pre_manifest = safe_export('pre_tau8_checkpoint')


In [ ]:
# Cell 4 - Prepare local frozen files and inspect baseline winner

for name, target in TARGETS.items():
    target['artifact_dir'].mkdir(parents=True, exist_ok=True)
    if not target['local_frozen'].exists():
        assert target['drive_frozen'].exists(), f'Missing frozen artifact: {target["drive_frozen"]}'
        shutil.copy2(target['drive_frozen'], target['local_frozen'])
        print(f'[state:{name}] copied frozen -> {target["local_frozen"]}')
    assert target['corpus_dir'].exists(), f'Missing corpus: {target["corpus_dir"]}'

leaderboard = load_json(LAB_ROOT / 'leaderboard.json', {'rows': []}).get('rows', [])
leaderboard = sorted(
    leaderboard,
    key=lambda r: (float(r.get('offline_projected_tps') or 0.0), float(r.get('tau') or 0.0)),
    reverse=True,
)
print('top leaderboard rows:')
for r in leaderboard[:10]:
    print(f"  {r.get('target')} {r.get('tag')} tau={r.get('tau')} accepted={r.get('accepted_draft_tokens_per_verify')} tps={r.get('offline_projected_tps')} head={r.get('head')}")


In [ ]:
# Cell 5 - Tau8 rollout fine-tune and eval from q1p5 winner

from safetensors import safe_open


def read_head_meta(head_path):
    meta = {}
    try:
        with safe_open(str(head_path), framework='pt', device='cpu') as f:
            meta = f.metadata() or {}
    except Exception as e:
        print(f'[tau8] WARN metadata read failed for {head_path}: {e}')
    return meta


def baseline_rows_for(name, k):
    rows = load_json(LAB_ROOT / 'leaderboard.json', {'rows': []}).get('rows', [])
    rows = [r for r in rows if r.get('target') == name and r.get('head')]
    rows.sort(key=lambda r: (float(r.get('offline_projected_tps') or 0.0), float(r.get('tau') or 0.0)), reverse=True)
    return rows[:k]


def copy_warm_start(src_ckpt_dir, dst_ckpt_dir, retries=3):
    src_latest = Path(src_ckpt_dir) / 'latest.npz'
    dst_ckpt_dir = Path(dst_ckpt_dir)
    dst_latest = dst_ckpt_dir / 'latest.npz'
    if dst_latest.exists():
        return True
    if not src_latest.exists():
        print(f'[tau8] WARN no warm-start latest.npz at {src_latest}; training cold')
        return False
    dst_ckpt_dir.mkdir(parents=True, exist_ok=True)
    last_error = None
    for attempt in range(1, retries + 1):
        tmp = dst_latest.with_suffix(dst_latest.suffix + f'.tmp.{attempt}')
        try:
            if tmp.exists():
                tmp.unlink()
            with open(src_latest, 'rb') as fsrc, open(tmp, 'wb') as fdst:
                shutil.copyfileobj(fsrc, fdst, length=16 * 1024 * 1024)
                fdst.flush()
                os.fsync(fdst.fileno())
            shutil.copystat(src_latest, tmp)
            os.replace(tmp, dst_latest)
            print(f'[tau8] warm-start copied {src_latest} -> {dst_latest}')
            return True
        except OSError as e:
            last_error = e
            print(f'[tau8] warm-start copy failed attempt {attempt}/{retries}: {e}')
            try:
                if tmp.exists():
                    tmp.unlink()
            except Exception:
                pass
            if getattr(e, 'errno', None) == 107 or 'Transport endpoint is not connected' in str(e):
                remount_drive_for_export()
            time.sleep(min(20, 2 * attempt))
    raise OSError(f'warm-start copy failed after {retries} attempts: {src_latest} -> {dst_latest}: {last_error}')


def eval_head(name, head):
    target = TARGETS[name]
    head = Path(head)
    tag = head.parent.name if head.parent.name != 'heads' else head.stem
    out_dir = Path(target['artifact_dir']) / 'eval' / tag
    out_dir.mkdir(parents=True, exist_ok=True)
    tau_path = out_dir / 'tau.json'
    frontier_path = out_dir / 'frontier.json'
    meta = read_head_meta(head)
    nb = meta.get('num_blocks', '1')
    hh = meta.get('n_heads', '16')
    ff = meta.get('ff_mult', '4.0')
    if not tau_path.exists():
        run_with_heartbeat([
            sys.executable, 'colab/eagle5_tau_eval_pytorch.py',
            '--ckpt', str(head),
            '--frozen', str(target['local_frozen']),
            '--corpus', str(target['corpus_dir']),
            '--out', str(tau_path),
            '--depth', str(TAU_DEPTH),
            '--max-windows', str(target['eval_max_windows']),
            '--max-row-tokens', str(target['train_max_row_tokens']),
            '--num-blocks', str(nb),
            '--head-heads', str(hh),
            '--head-ff-mult', str(ff),
            '--base-tps', str(target['base_tps_placeholder']),
            '--w4a8-multiplier', '1.0',
            '--spec-efficiency', str(target['spec_efficiency_placeholder']),
        ], label=f'tau8-tau-{name}-{tag}', interval_sec=60)
    if not frontier_path.exists():
        run_with_heartbeat([
            sys.executable, 'colab/eagle5_frontier_policy.py',
            '--ckpt', str(head),
            '--frozen', str(target['local_frozen']),
            '--corpus', str(target['corpus_dir']),
            '--out', str(frontier_path),
            '--max-depth', str(target['frontier_max_depth']),
            '--depths', FRONTIER_DEPTHS,
            '--lattice-widths', FRONTIER_WIDTHS,
            '--max-windows', str(target['eval_max_windows']),
            '--max-row-tokens', str(target['train_max_row_tokens']),
            '--eval-batch-size', '192',
            '--num-blocks', str(nb),
            '--head-heads', str(hh),
            '--head-ff-mult', str(ff),
            '--base-tps', str(target['base_tps_placeholder']),
            '--w4a8-multiplier', '1.0',
            '--spec-efficiency', str(target['spec_efficiency_placeholder']),
        ], label=f'tau8-frontier-{name}-{tag}', interval_sec=60)
    tau = load_json(tau_path, {})
    frontier = load_json(frontier_path, {})
    best = frontier.get('policies', {}).get('best_deployable', {})
    overall = frontier.get('policies', {}).get('best_overall', {})
    return {
        'target': name,
        'tag': tag,
        'head': str(head),
        'tau_path': str(tau_path),
        'frontier_path': str(frontier_path),
        'tau': tau.get('tau'),
        'depth1_accept_rate': tau.get('depth1_accept_rate'),
        'best_deployable': best,
        'best_overall': overall,
        'offline_projected_tps': best.get('projected_dec_tps', 0.0),
        'accepted_draft_tokens_per_verify': best.get('accepted_draft_tokens_per_verify', 0.0),
        'policy_kind': best.get('kind'),
        'metadata': meta,
    }

TAU8_HEADS = {name: [] for name in TAU8_TARGET_ORDER}
if RUN_TAU8:
    for name in TAU8_TARGET_ORDER:
        target = TARGETS[name]
        baselines = baseline_rows_for(name, TAU8_TOP_BASELINES_PER_TARGET)
        if not baselines:
            raise RuntimeError(f'no leaderboard baseline head for {name}')
        for base_rank, base in enumerate(baselines, start=1):
            base_head = Path(base['head'])
            base_tag = base_head.parent.name
            meta = read_head_meta(base_head)
            nb = int(meta.get('num_blocks', '1'))
            hh = int(meta.get('n_heads', '16'))
            ff = float(meta.get('ff_mult', '4.0'))
            for spec in TAU8_SPECS:
                tag = f'{name}_tau8_{spec["name"]}_from_{base_tag}'
                ckpt_dir = Path(target['artifact_dir']) / 'checkpoints' / tag
                head = ckpt_dir / 'head_final.safetensors'
                if head.exists():
                    print(f'[tau8:{name}] skip existing {head}')
                    TAU8_HEADS[name].append(head)
                    continue
                copy_warm_start(base_head.parent, ckpt_dir)
                batch = 56 if BIG_GPU else 24
                cmd = [
                    sys.executable, '-u', 'colab/eagle5_train_pytorch.py',
                    '--corpus-dir', str(target['corpus_dir']),
                    '--frozen', str(target['local_frozen']),
                    '--ckpt-dir', str(ckpt_dir),
                    '--epochs', str(spec['epochs']),
                    '--batch-size', str(batch),
                    '--seq-len', '16',
                    '--lr', str(spec['lr']),
                    '--num-blocks', str(nb),
                    '--head-heads', str(hh),
                    '--head-ff-mult', str(ff),
                    '--capture-layer', str(target['capture_layer']),
                    '--max-rows', str(target['train_max_rows']),
                    '--max-row-tokens', str(target['train_max_row_tokens']),
                    '--sparsity-head', 'off',
                    '--seed', str(1000 + base_rank),
                    '--calib-loss-weight', str(spec['calib_loss_weight']),
                    '--residual-delta-loss-weight', str(spec['residual_delta_loss_weight']),
                    '--rollout-loss-weight', str(spec['rollout_loss_weight']),
                    '--rollout-depth', str(spec['rollout_depth']),
                    '--rollout-starts-per-batch', str(spec['rollout_starts_per_batch']),
                    '--rollout-draft-prob', str(spec['rollout_draft_prob']),
                    '--rollout-depth-gamma', str(spec['rollout_depth_gamma']),
                    '--save-safetensors',
                ]
                print(f'\n=== [tau8:{name}] train {tag} batch={batch} warm={base_tag}')
                run_with_heartbeat(cmd, label=f'tau8_{name}_{spec["name"]}', interval_sec=60)
                if not head.exists():
                    raise FileNotFoundError(f'tau8 head missing after train: {head}')
                TAU8_HEADS[name].append(head)
else:
    print('RUN_TAU8=False; collecting existing tau8 heads only.')
    for name in TAU8_TARGET_ORDER:
        ckpt_root = Path(TARGETS[name]['artifact_dir']) / 'checkpoints'
        TAU8_HEADS[name] = sorted(p for p in ckpt_root.glob(f'{name}_tau8_*/head_final.safetensors'))

TAU8_ROWS = []
if RUN_TAU8_EVAL:
    for name, heads in TAU8_HEADS.items():
        for head in heads:
            TAU8_ROWS.append(eval_head(name, head))
else:
    print('RUN_TAU8_EVAL=False; skipping tau/frontier eval.')

if TAU8_ROWS:
    existing_rows = load_json(LAB_ROOT / 'leaderboard.json', {}).get('rows', [])
    by_key = {(r.get('target'), r.get('tag')): r for r in existing_rows}
    for row in TAU8_ROWS:
        by_key[(row.get('target'), row.get('tag'))] = row
    merged = sorted(by_key.values(), key=lambda r: (float(r.get('offline_projected_tps') or 0.0), float(r.get('tau') or 0.0)), reverse=True)
    write_json_atomic(LAB_ROOT / 'leaderboard.json', {'schema': 'dismantle-maximal-spec-leaderboard-v1', 'rows': merged})
    print('\n=== Tau8 rows ===')
    for r in TAU8_ROWS:
        print(f"{r['target']:7s} {r['tag'][:72]:72s} tau={r.get('tau')} acc1={r.get('depth1_accept_rate')} accepted={r.get('accepted_draft_tokens_per_verify')} offline_tps={r.get('offline_projected_tps')} policy={r.get('policy_kind')}")


In [ ]:
# Cell 6 - Train-on-best continuation ladder

# This stage milks the strongest evaluated head instead of extending every
# candidate equally. Each round re-reads leaderboard.json, picks the current
# best q1p5 row with a resumable latest.npz, warm-starts a new checkpoint, then
# evaluates and merges the result before the next round.

BEST_CONTINUE_SPECS = [
    {
        'name': 'polish_d8_w006_p060_g095_lr5e-5',
        'epochs': 2,
        'lr': 5e-5,
        'seq_len': 16,
        'rollout_loss_weight': 0.06,
        'rollout_depth': 8,
        'rollout_starts_per_batch': 3,
        'rollout_draft_prob': 0.60,
        'rollout_depth_gamma': 0.95,
        'calib_loss_weight': 0.10,
        'residual_delta_loss_weight': 0.008,
    },
    {
        'name': 'push_d8_w010_p080_g092_lr4e-5',
        'epochs': 2,
        'lr': 4e-5,
        'seq_len': 16,
        'rollout_loss_weight': 0.10,
        'rollout_depth': 8,
        'rollout_starts_per_batch': 3,
        'rollout_draft_prob': 0.80,
        'rollout_depth_gamma': 0.92,
        'calib_loss_weight': 0.12,
        'residual_delta_loss_weight': 0.010,
    },
]


def leaderboard_sort_key(row):
    return (
        float(row.get('offline_projected_tps') or 0.0),
        float(row.get('accepted_draft_tokens_per_verify') or 0.0),
        float(row.get('tau') or 0.0),
    )


def merge_leaderboard_rows(rows_to_merge):
    if not rows_to_merge:
        return []
    existing_rows = load_json(LAB_ROOT / 'leaderboard.json', {}).get('rows', [])
    by_key = {(r.get('target'), r.get('tag')): r for r in existing_rows}
    for row in rows_to_merge:
        by_key[(row.get('target'), row.get('tag'))] = row
    merged = sorted(by_key.values(), key=leaderboard_sort_key, reverse=True)
    write_json_atomic(LAB_ROOT / 'leaderboard.json', {'schema': 'dismantle-maximal-spec-leaderboard-v1', 'rows': merged})
    return merged


def best_resumable_row(name):
    rows = load_json(LAB_ROOT / 'leaderboard.json', {'rows': []}).get('rows', [])
    candidates = []
    for row in rows:
        if row.get('target') != name or not row.get('head'):
            continue
        head = Path(row['head'])
        latest = head.parent / 'latest.npz'
        if head.exists() and latest.exists():
            candidates.append(row)
        elif head.exists():
            print(f'[best] skip non-resumable row without latest.npz: {head}')
    candidates.sort(key=leaderboard_sort_key, reverse=True)
    if not candidates:
        raise RuntimeError(f'no resumable leaderboard head for {name}')
    return candidates[0]


def train_best_continuation(name, round_idx, base_row, spec):
    target = TARGETS[name]
    base_head = Path(base_row['head'])
    base_tag = base_head.parent.name
    base_hash = hashlib.sha1(base_tag.encode()).hexdigest()[:8]
    tag = f'{name}_best_r{round_idx}_{spec["name"]}_from_{base_hash}'
    ckpt_dir = Path(target['artifact_dir']) / 'checkpoints' / tag
    head = ckpt_dir / 'head_final.safetensors'
    if head.exists():
        print(f'[best:{name}] skip existing {head}')
        return head, tag

    meta = read_head_meta(base_head)
    nb = int(meta.get('num_blocks', '1'))
    hh = int(meta.get('n_heads', '16'))
    ff = float(meta.get('ff_mult', '4.0'))
    copy_warm_start(base_head.parent, ckpt_dir)
    batch = 56 if BIG_GPU else 24
    cmd = [
        sys.executable, '-u', 'colab/eagle5_train_pytorch.py',
        '--corpus-dir', str(target['corpus_dir']),
        '--frozen', str(target['local_frozen']),
        '--ckpt-dir', str(ckpt_dir),
        '--epochs', str(spec['epochs']),
        '--batch-size', str(batch),
        '--seq-len', str(spec.get('seq_len', 16)),
        '--lr', str(spec['lr']),
        '--num-blocks', str(nb),
        '--head-heads', str(hh),
        '--head-ff-mult', str(ff),
        '--capture-layer', str(target['capture_layer']),
        '--max-rows', str(target['train_max_rows']),
        '--max-row-tokens', str(target['train_max_row_tokens']),
        '--sparsity-head', 'off',
        '--seed', str(7000 + round_idx),
        '--calib-loss-weight', str(spec['calib_loss_weight']),
        '--residual-delta-loss-weight', str(spec['residual_delta_loss_weight']),
        '--rollout-loss-weight', str(spec['rollout_loss_weight']),
        '--rollout-depth', str(spec['rollout_depth']),
        '--rollout-starts-per-batch', str(spec['rollout_starts_per_batch']),
        '--rollout-draft-prob', str(spec['rollout_draft_prob']),
        '--rollout-depth-gamma', str(spec['rollout_depth_gamma']),
        '--save-safetensors',
    ]
    print(f'\n=== [best:{name}] round={round_idx} train {tag}')
    print(f"[best:{name}] warm={base_tag} tau={base_row.get('tau')} accepted={base_row.get('accepted_draft_tokens_per_verify')} tps={base_row.get('offline_projected_tps')}")
    run_with_heartbeat(cmd, label=f'best_{name}_r{round_idx}_{spec["name"]}', interval_sec=60)
    if not head.exists():
        raise FileNotFoundError(f'best continuation head missing after train: {head}')
    return head, tag


BEST_CONTINUE_ROWS = []
BEST_CONTINUE_HEADS = {name: [] for name in TAU8_TARGET_ORDER}

if RUN_TRAIN_ON_BEST:
    for round_idx in range(1, int(BEST_CONTINUE_ROUNDS) + 1):
        spec = BEST_CONTINUE_SPECS[min(round_idx - 1, len(BEST_CONTINUE_SPECS) - 1)]
        print(f'\n=== Best-continuation round {round_idx}/{BEST_CONTINUE_ROUNDS}: {spec["name"]} ===')
        for name in TAU8_TARGET_ORDER:
            base_row = best_resumable_row(name)
            head, tag = train_best_continuation(name, round_idx, base_row, spec)
            BEST_CONTINUE_HEADS[name].append(head)
            if RUN_TRAIN_ON_BEST_EVAL:
                row = eval_head(name, head)
                row['source_tag'] = Path(base_row['head']).parent.name
                row['source_tau'] = base_row.get('tau')
                row['source_offline_projected_tps'] = base_row.get('offline_projected_tps')
                row['best_continue_round'] = round_idx
                row['best_continue_spec'] = spec['name']
                BEST_CONTINUE_ROWS.append(row)
                merged = merge_leaderboard_rows([row])
                top = merged[0]
                print(f"[best] leaderboard top now: {top.get('target')} {top.get('tag')} tau={top.get('tau')} accepted={top.get('accepted_draft_tokens_per_verify')} tps={top.get('offline_projected_tps')}")
else:
    print('RUN_TRAIN_ON_BEST=False; collecting existing best-continuation heads only.')
    for name in TAU8_TARGET_ORDER:
        ckpt_root = Path(TARGETS[name]['artifact_dir']) / 'checkpoints'
        BEST_CONTINUE_HEADS[name] = sorted(p for p in ckpt_root.glob(f'{name}_best_r*/head_final.safetensors'))

if RUN_TRAIN_ON_BEST_EVAL and not RUN_TRAIN_ON_BEST:
    for name, heads in BEST_CONTINUE_HEADS.items():
        for head in heads:
            row = eval_head(name, head)
            BEST_CONTINUE_ROWS.append(row)
    merge_leaderboard_rows(BEST_CONTINUE_ROWS)

if BEST_CONTINUE_ROWS:
    print('\n=== Best-continuation rows ===')
    for r in BEST_CONTINUE_ROWS:
        print(f"{r['target']:7s} {r['tag'][:72]:72s} tau={r.get('tau')} accepted={r.get('accepted_draft_tokens_per_verify')} offline_tps={r.get('offline_projected_tps')} policy={r.get('policy_kind')} from={r.get('source_tag')}")


In [ ]:
# Cell 7 - Safe export again after tau8/best rows have been merged

post_manifest = safe_export('post_tau8_checkpoint')
print('Done. Download or mirror this folder locally:')
print(EXPORT_ROOT)
